# Lab 7: Observabilidade e Governance (10 min)

## Objetivos
- Entender a importância da observabilidade em apps de IA
- Ativar tracing com Azure AI Inference SDK
- Ver métricas e logs
- Compreender governance e content safety

## 📖 Porquê Observabilidade?

Aplicações de IA são **não determinísticas** — a mesma pergunta pode gerar respostas diferentes. Precisamos de:

| Pilar | O que monitorizar | Ferramenta |
|-------|-------------------|------------|
| **Tracing** | Cada chamada ao modelo (input/output) | Application Insights |
| **Métricas** | Latência, tokens, erros, custos | Azure Monitor |
| **Logs** | Conversas, feedback dos utilizadores | App Insights / Log Analytics |
| **Avaliação** | Qualidade das respostas ao longo do tempo | Azure AI Evaluation |

### Governance

| Aspecto | Descrição |
|---------|----------|
| **Content Safety** | Filtros para conteúdo ofensivo, violento, etc. |
| **RBAC** | Quem tem acesso a quê |
| **Data Residency** | Onde os dados são processados |
| **Audit Logs** | Registo de todas as operações |
| **Responsible AI** | Dashboard de métricas éticas |

In [ ]:
# Setup
from dotenv import load_dotenv
import os
import time
import json

load_dotenv("../.env")

from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
print("✅ Setup concluído!")

## 💻 Exercício 7.1: Implementar Logging Manual

Antes de ferramentas sofisticadas, vamos implementar observabilidade básica — logar cada chamada.

In [ ]:
# Wrapper que adiciona observabilidade a cada chamada
import datetime

logs = []  # Em produção, isto iria para Application Insights

def chamar_modelo_com_observabilidade(mensagens: list, **kwargs) -> dict:
    """Wrapper que loga cada chamada ao modelo."""
    inicio = time.time()
    erro = None
    resposta_texto = None
    tokens = {}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=mensagens,
            **kwargs,
        )
        resposta_texto = response.choices[0].message.content
        tokens = {
            "prompt": response.usage.prompt_tokens,
            "completion": response.usage.completion_tokens,
            "total": response.usage.total_tokens,
        }
    except Exception as e:
        erro = str(e)
    
    duracao = time.time() - inicio
    
    log_entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "model": DEPLOYMENT,
        "input": mensagens[-1]["content"][:100],
        "output": resposta_texto[:100] if resposta_texto else None,
        "tokens": tokens,
        "duration_s": round(duracao, 3),
        "error": erro,
        "status": "success" if not erro else "error",
    }
    
    logs.append(log_entry)
    return {"response": resposta_texto, "log": log_entry}

print("✅ Função com observabilidade pronta!")

In [ ]:
# Fazer algumas chamadas para gerar dados de observabilidade
perguntas = [
    "Qual é a capital de Portugal?",
    "Explica o que é machine learning em 2 frases.",
    "Quais são os 3 maiores rios de Portugal?",
    "O que é uma API REST?",
    "Conta uma piada sobre programadores.",
]

for p in perguntas:
    resultado = chamar_modelo_com_observabilidade(
        mensagens=[{"role": "user", "content": p}],
        max_tokens=100,
        temperature=0.7,
    )
    print(f"✅ {p[:40]}... ({resultado['log']['duration_s']}s, {resultado['log']['tokens'].get('total', 0)} tokens)")

In [ ]:
# Analisar os logs - dashboard simples
import pandas as pd

df = pd.DataFrame(logs)

print("📊 DASHBOARD DE OBSERVABILIDADE")
print("=" * 50)
print(f"\n📈 Total de chamadas: {len(df)}")
print(f"✅ Chamadas com sucesso: {len(df[df['status'] == 'success'])}")
print(f"❌ Chamadas com erro: {len(df[df['status'] == 'error'])}")

# Extrair tokens do dict
df['total_tokens'] = df['tokens'].apply(lambda x: x.get('total', 0) if isinstance(x, dict) else 0)

print(f"\n⏱️ Latência média: {df['duration_s'].mean():.3f}s")
print(f"⏱️ Latência máxima: {df['duration_s'].max():.3f}s")
print(f"⏱️ Latência mínima: {df['duration_s'].min():.3f}s")

print(f"\n🔢 Total tokens consumidos: {df['total_tokens'].sum()}")
print(f"🔢 Média tokens por chamada: {df['total_tokens'].mean():.0f}")

# Estimativa de custo (GPT-4o approx pricing)
custo_estimado = df['total_tokens'].sum() / 1000 * 0.005
print(f"\n💰 Custo estimado: ${custo_estimado:.4f} USD")

## 📖 Exercício 7.2: Content Safety Filters

O Azure OpenAI tem **filtros de conteúdo** built-in que bloqueiam categorias como:
- 🔴 Hate speech
- 🔴 Violência
- 🔴 Conteúdo sexual
- 🔴 Self-harm

Vamos verificar os filtros em ação.

In [ ]:
# Verificar content filter annotations numa resposta normal
response = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{"role": "user", "content": "Diz-me 3 benefícios da meditação."}],
    max_tokens=150,
)

print("🤖 Resposta:")
print(response.choices[0].message.content)

# Verificar filtros (se disponíveis no response)
if hasattr(response.choices[0], 'content_filter_results'):
    filtros = response.choices[0].content_filter_results
    print(f"\n🛡️ Content Filters:")
    if filtros:
        for filtro_nome, filtro_val in vars(filtros).items():
            if filtro_val and hasattr(filtro_val, 'severity'):
                print(f"   {filtro_nome}: {filtro_val.severity}")
else:
    print("\n🛡️ Content filters ativos (detalhes no portal Azure)")

print("\n💡 Os filtros funcionam automaticamente — não precisas de fazer nada!")

## 🖥️ Exercício 7.3: Ver Observabilidade no Portal

1. Vai ao [Portal Foundry](https://ai.azure.com) → o teu projeto
2. No menu lateral, clica em **Tracing**
3. Observa:
   - As chamadas que fizeste nos labs anteriores
   - Latência de cada chamada
   - Tokens consumidos
   - Input/Output de cada chamada

4. Vai a **Metrics** e observa:
   - Token usage over time
   - Request count
   - Latency percentiles

## 📖 Governance - Best Practices

| Prática | Como implementar |
|---------|------------------|
| **RBAC** | Usar roles Azure para acesso ao projeto (Reader, Contributor, Owner) |
| **Key Rotation** | Rodar chaves regularmente, usar Managed Identity quando possível |
| **Data Residency** | Escolher região do deployment (ex: West Europe para GDPR) |
| **Audit Logs** | Activity Log no portal Azure registra todas as operações |
| **Content Filters** | Manter filtros ativos, personalizar se necessário |
| **Responsible AI** | Usar Azure AI Content Safety para testes adicionais |

## ✅ Resumo

Neste lab aprendeste:
- Porquê a observabilidade é crítica em apps de IA
- Como implementar logging básico de chamadas
- Como analisar métricas (latência, tokens, custos)
- Content Safety filters do Azure OpenAI
- Best practices de governance

**Próximo:** [Lab 8 - Red Teaming](lab08-red-teaming.ipynb)